# PINN 입문 워크스루: 1D 감쇠 조화 진동자 (PyTorch)

이 노트북은 **PINN (Physics-Informed Neural Network)** 의 작동 원리를 셀 단위로 뜯어보며 직접 실험해보기 위한 학습용 노트북입니다.

### 이 노트북에서 배우는 것
1. 왜 일반 신경망(NN)은 데이터가 없는 구간(외삽, Extrapolation)에서 실패하는가?
2. PyTorch `torch.autograd.grad`로 신경망 출력의 1차/2차 미분($u'$, $u''$)을 어떻게 구하는가?
3. 물리 방정식 잔차(Physics Residual)를 손실 함수에 추가하면 예측이 어떻게 달라지는가?


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# 프로젝트의 src/ 폴더를 import 경로에 추가
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root / "src"))

from pinn import DampedHarmonicOscillator, MLP, derivative

torch.manual_seed(42)
print("PyTorch version:", torch.__version__)


## Step 1. 물리 문제 정의 및 데이터 준비

우리가 풀려는 물리 법칙은 **1D 감쇠 조화 진동자**입니다:

$$\frac{d^2 u}{dt^2} + \mu \frac{du}{dt} + k u = 0 \qquad (\mu = 4,\; k = 400)$$

- **관측 데이터**: $t \in [0, 0.4]$ 구간에서만 딱 **10개**의 점만 관측했다고 가정합니다.
- **목표**: 관측 데이터가 전혀 없는 미래 구간 $t \in [0.4, 1.0]$ 의 움직임까지 정확히 맞추기!


In [ ]:
problem = DampedHarmonicOscillator(delta=2.0, omega_0=20.0)

# 전체 구간 정답 (평가 및 시각화용)
t_test = torch.linspace(0.0, 1.0, 500).view(-1, 1)
u_exact = problem.exact_solution(t_test)

# 소량의 관측 데이터 (앞구간 t ∈ [0, 0.4] 에서 10개)
t_data = torch.linspace(0.0, 0.4, 10).view(-1, 1)
u_data = problem.exact_solution(t_data)

# 물리 법칙 검사 지점 (전체 구간 t ∈ [0, 1] 에서 40개, 정답 u 값은 주지 않음!)
t_physics = torch.linspace(0.0, 1.0, 40).view(-1, 1).requires_grad_(True)

plt.figure(figsize=(8, 3.5))
plt.plot(t_test, u_exact, "--", color="black", label="Exact solution")
plt.scatter(t_data, u_data, color="tab:red", zorder=5, label="Observed data (10 pts)")
plt.axvspan(0.4, 1.0, color="gray", alpha=0.15, label="No-data region")
plt.xlabel("Time t")
plt.ylabel("u(t)")
plt.title("Problem Setup: Only 10 Data Points in t in [0, 0.4]")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Step 2. 일반 신경망(Standard NN) 먼저 학습해보기

물리 법칙 없이 오직 10개의 관측 데이터만 `MSELoss` 로 학습하면 어떻게 될까요?


In [ ]:
torch.manual_seed(42)
nn_model = MLP(in_dim=1, out_dim=1, hidden_dim=32, num_hidden_layers=3)
optimizer = torch.optim.Adam(nn_model.parameters(), lr=1e-3)
mse = nn.MSELoss()

for epoch in range(1, 3001):
    optimizer.zero_grad()
    loss = mse(nn_model(t_data), u_data)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    u_nn = nn_model(t_test)

plt.figure(figsize=(8, 3.5))
plt.plot(t_test, u_exact, "--", color="black", label="Exact")
plt.plot(t_test, u_nn, color="tab:red", linewidth=2, label="Standard NN")
plt.scatter(t_data, u_data, color="navy", zorder=5, label="Data (10 pts)")
plt.title("Standard NN: Fits Data, Fails Completely on Future (t > 0.4)")
plt.ylim(-1.3, 1.3)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Step 3. PINN의 핵심: 자동미분(Autograd) 확인하기

`derivative(u, t, order=1)` 와 `derivative(u, t, order=2)` 가 실제로 $du/dt$, $d^2u/dt^2$ 를 계산하는지 간단한 함수로 먼저 눈으로 확인해봅시다.


In [ ]:
t_check = torch.tensor([[0.0], [1.0], [2.0]], requires_grad=True)
y_check = t_check ** 3  # y = t^3 -> y' = 3t^2, y'' = 6t

print("t      :", t_check.detach().squeeze().tolist())
print("y = t^3:", y_check.detach().squeeze().tolist())
print("dy/dt  :", derivative(y_check, t_check, order=1).detach().squeeze().tolist())
print("d2y/dt2:", derivative(y_check, t_check, order=2).detach().squeeze().tolist())


## Step 4. PINN 학습하기 (Data Loss + Physics Loss)

이제 손실 함수에 **물리 방정식 잔차(Residual)** 를 추가합니다:

```python
residual = problem.physics_residual(pinn_model, t_physics)  # u'' + mu*u' + k*u
loss_physics = torch.mean(residual ** 2)
loss = loss_data + lambda_physics * loss_physics
```

아래 셀을 실행한 뒤, `lambda_physics` 값을 바꿔가며 결과가 어떻게 변하는지 직접 실험해보세요!


In [ ]:
torch.manual_seed(42)
pinn_model = MLP(in_dim=1, out_dim=1, hidden_dim=32, num_hidden_layers=3)
optimizer = torch.optim.Adam(pinn_model.parameters(), lr=1e-3)
mse = nn.MSELoss()

lambda_physics = 1e-4

for epoch in range(1, 12001):
    optimizer.zero_grad()

    # 1) 관측 데이터 손실
    loss_data = mse(pinn_model(t_data), u_data)

    # 2) 물리 법칙 손실 (정답 u 없이 ODE 식만 만족하도록 강제!)
    residual = problem.physics_residual(pinn_model, t_physics)
    loss_physics = torch.mean(residual ** 2)

    # 3) 총 손실
    loss = loss_data + lambda_physics * loss_physics
    loss.backward()
    optimizer.step()

    if epoch % 3000 == 0:
        print(f"Epoch {epoch:5d} | Data: {loss_data.item():.6f} | Physics(weighted): {(lambda_physics * loss_physics).item():.6f}")

with torch.no_grad():
    u_pinn = pinn_model(t_test)

plt.figure(figsize=(8, 3.5))
plt.plot(t_test, u_exact, "--", color="black", label="Exact")
plt.plot(t_test, u_pinn, color="tab:blue", linewidth=2, label="PINN")
plt.scatter(t_data, u_data, color="navy", zorder=5, label="Data (10 pts)")
plt.axvspan(0.4, 1.0, color="green", alpha=0.1, label="Extrapolation zone")
plt.title("PINN: Accurately Predicts the Future Using Physics Laws!")
plt.ylim(-1.3, 1.3)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
